# Guía de Pandas

Referencia de **todas las funciones y métodos de pandas** que necesitan para armar el proyecto, ordenados según el flujo del ETL (extraer → perfilar → limpiar → transformar → agrupar → cargar).

> Los ejemplos usan **datos de juguete inventados** (no son sus datasets). Lo que hay que hacer con vgsales, anime y Goodreads queda para ustedes: cada sección termina con un bloque **Para tu proyecto** que dice qué aplicar, sin resolverlo.

**Índice**
1. Cargar datos
2. Perfilar (conocer el dataset)
3. Seleccionar y filtrar
4. Limpieza y tipos
5. Texto (`.str`)
6. Columnas con varios valores (listas, `explode`, dummies)
7. Vocabulario común y tabla de mapeo
8. Agrupar y agregar (stands = productor)
9. Unir tablas (`merge`, `concat`)
10. Frecuencias, IDF y ordenar
11. Datos sintéticos (asistentes, interacciones)
12. Salida: guardar y preparar para MongoDB
13. Mapa: qué función va en qué paso del proyecto

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

print(pd.__version__)
pd.set_option("display.max_columns", 30)   # mostrar más columnas
pd.set_option("display.max_colwidth", 60)  # ancho de celdas de texto
pd.set_option("display.width", 160)

## 0. Datos de juguete para los ejemplos
Imitan la *forma* de sus fuentes (un productor, géneros en texto, nulos, duplicados, listas), no su contenido.

In [ ]:
toy = pd.DataFrame({
    "Name":      ["Alpha", "Beta", "Gamma", "Gamma", "Delta", "Epsilon", None],
    "Publisher": [" Acme ", "acme", "Zeta Co", "Zeta Co", "Omega", None, "Omega"],
    "Year":      [2001, 2005, "2010", "2010", np.nan, 1999, 2020],
    "Genres":    ["Action, Fantasy", "Action", "Sci-Fi, Drama, Action", "Sci-Fi, Drama, Action", None, "Fantasy", "Drama"],
    "Sales":     [1.5, 0.3, 9.9, 9.9, 0.1, 2.2, 0.7],
})
toy

## 1. Cargar datos
| Función | Para qué |
|---|---|
| `pd.read_csv` | CSV, también `.csv.zip` / `.gz` (compresión se infiere) |
| `pd.read_json(lines=True)` | JSON Lines (Goodreads), `.json.gz` |
| `pd.read_excel`, `pd.read_parquet` | otros formatos |
| `pd.read_json(chunksize=...)` / `read_csv(chunksize=...)` | archivos que no caben en memoria |

In [ ]:
RAIZ = Path.cwd()
RAW = RAIZ / "datos" / "raw"

# Parámetros útiles de read_csv (se muestran sin ejecutar porque dependen de sus archivos):
#
# pd.read_csv(RAW / "archivo.csv",
#     usecols=["col1", "col2"],      # leer solo algunas columnas (ahorra memoria)
#     dtype={"col1": "string"},      # forzar tipos
#     nrows=1000,                    # solo las primeras N filas (para explorar)
#     encoding="utf-8",              # o "latin-1" si hay caracteres raros
#     sep=",",                       # separador
#     na_values=["N/A", "-", ""],    # qué texto cuenta como nulo
#     parse_dates=["fecha"],         # convertir a datetime al leer
# )
#
# JSON Lines (un objeto JSON por línea), comprimido:
# pd.read_json(RAW / "archivo.json.gz", lines=True, nrows=1000)
#
# Archivo grande leído por pedazos:
# for trozo in pd.read_json(ruta, lines=True, chunksize=50_000):
#     procesar(trozo)

# Demostración con un CSV en memoria:
from io import StringIO
csv_demo = StringIO("a,b\n1,x\n2,N/A\n3,z")
pd.read_csv(csv_demo, na_values=["N/A"])

## 2. Perfilar: conocer el dataset

In [ ]:
print(toy.shape)            # (filas, columnas)
print(toy.columns.tolist()) # nombres de columnas
print(toy.dtypes)           # tipo de cada columna
toy.info()                  # tipos + no-nulos + memoria
toy.head(3)                 # primeras filas  (tail(), sample() también)

In [ ]:
toy.describe()                    # estadísticas de columnas numéricas
toy.describe(include="all")       # incluye texto
toy.isna().sum()                  # nulos por columna
toy.isna().mean().round(2)        # % de nulos por columna
toy.duplicated().sum()            # filas duplicadas exactas
toy.duplicated(subset=["Name"]).sum()   # duplicados según una columna
toy.nunique()                     # valores distintos por columna
toy["Publisher"].value_counts()               # frecuencia de cada valor
toy["Publisher"].value_counts(dropna=False)   # contando también los nulos
toy["Publisher"].value_counts(normalize=True) # proporciones
toy["Genres"].unique()            # lista de valores distintos
toy.memory_usage(deep=True)       # memoria por columna

## 3. Seleccionar y filtrar

In [ ]:
toy["Name"]                 # una columna (Series)
toy[["Name", "Publisher"]]  # varias columnas (DataFrame)
toy.loc[0:2, ["Name", "Year"]]   # por etiqueta (el fin INCLUYE)
toy.iloc[0:2, 0:2]               # por posición (el fin NO incluye)
toy.loc[toy["Sales"] > 1]        # filtro booleano
toy[(toy["Sales"] > 1) & (toy["Year"].notna())]   # & (y)  | (o)  ~ (no), con paréntesis
toy[toy["Publisher"].isin(["Omega", "Zeta Co"])]  # pertenece a una lista
toy[toy["Sales"].between(0.5, 3)]                 # rango
toy.query("Sales > 1 and Publisher == 'Omega'")   # filtro como texto
toy.drop(columns=["Sales"])       # quitar columnas
toy.filter(like="Gen")            # columnas cuyo nombre contiene "Gen"

## 4. Limpieza y tipos

In [ ]:
df = toy.copy()

df = df.dropna(subset=["Publisher"])                 # quitar filas con nulo en esa columna
df = df.drop_duplicates()                             # quitar duplicados exactos
df = df.drop_duplicates(subset=["Name"], keep="first")# duplicados por columna
df["Genres"] = df["Genres"].fillna("")               # rellenar nulos
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")   # texto -> número (lo malo pasa a NaN)
df["Year"] = df["Year"].astype("Int64")              # entero que admite nulos
df = df.rename(columns={"Name": "nombre", "Publisher": "productor"})
df = df.reset_index(drop=True)                       # reordenar índice tras filtrar
df

In [ ]:
# Otras herramientas
s = pd.Series(["a", "b", "c", "a"])
s.replace({"a": "A"})                         # reemplazar valores
s.map({"a": 1, "b": 2})                       # mapear con dict (lo no encontrado -> NaN)
s.apply(lambda x: x.upper())                  # función por elemento
s.astype("category")                          # tipo categoría (ahorra memoria)
pd.to_datetime(["2026-05-01 10:00", "2026-05-01 12:30"])   # texto -> fecha
pd.to_datetime("2026-05-01") + pd.Timedelta(hours=2)       # sumar tiempo
pd.cut(pd.Series([1, 5, 9]), bins=[0, 3, 6, 10], labels=["bajo", "medio", "alto"])  # discretizar
pd.Series([3, 1, 2]).clip(lower=2)            # limitar valores

## 5. Texto con `.str`


In [ ]:
t = pd.Series([" Role-Playing ", "SCI-FI", "Slice of Life", None, "Sports/Racing"])

t.str.strip()                 # quitar espacios en los bordes
t.str.lower()                 # minúsculas (upper, title, capitalize)
t.str.replace("-", " ", regex=False)   # reemplazo literal
t.str.replace(r"\s+", " ", regex=True) # reemplazo con regex
t.str.contains("sci", case=False, na=False)   # ¿contiene?  (na=False evita errores en filtros)
t.str.startswith("Sl", na=False)
t.str.len()
t.str.split("/")                       # texto -> lista
t.str.extract(r"(\w+)")               # captura con regex
t.str.cat(sep=" | ", na_rep="")        # unir en un solo texto

In [ ]:
# Quitar tildes (pandas no trae una función directa; se hace con unicodedata + map)
import unicodedata
def quitar_tildes(x):
    if not isinstance(x, str):
        return x
    return "".join(c for c in unicodedata.normalize("NFKD", x) if not unicodedata.combining(c))

pd.Series(["Acción", "Ciencia ficción"]).map(quitar_tildes)

# Encadenar pasos de normalización en una sola función reutilizable:
def normalizar(serie: pd.Series) -> pd.Series:
    return (serie.str.strip()
                 .str.lower()
                 .str.replace(r"[\s_]+", "-", regex=True))

normalizar(t)

## 6. Columnas con varios valores


In [ ]:
g = toy[["Name", "Genres"]].dropna().drop_duplicates().copy()

# 1) texto -> lista
g["lista"] = g["Genres"].str.split(",")
# 2) limpiar cada elemento de la lista (list comprehension dentro de apply/map)
g["lista"] = g["lista"].map(lambda xs: [x.strip().lower() for x in xs])
g

# 3) una fila por género
largo = g.explode("lista").rename(columns={"lista": "genero"})
largo

In [ ]:
# Variantes
g["Genres"].str.get_dummies(sep=", ")          # matriz 0/1 por género
pd.get_dummies(largo["genero"])                # dummies de una columna categórica
largo["genero"].value_counts()                 # frecuencia de cada género

# Si la columna viene como TEXTO que parece lista ("['a', 'b']") -> convertir:
import ast
pd.Series(["['a', 'b']", "['c']"]).map(ast.literal_eval)

# Listas de listas / dicts anidados (típico en JSON): 
anidado = pd.DataFrame({"id": [1, 2], "info": [{"x": 1, "y": 2}, {"x": 3, "y": 4}]})
pd.json_normalize(anidado["info"])             # dict -> columnas

## 7. Vocabulario común y tabla de mapeo
Idea: una tabla `origen → tag` que se aplica con `map` o `merge`.

In [ ]:
mapeo = pd.DataFrame({
    "origen": ["sci-fi", "science fiction", "role-playing", "fantasy", "action"],
    "tag":    ["ciencia-ficcion", "ciencia-ficcion", "rol-fantasia", "rol-fantasia", "accion"],
    "nivel":  ["general", "general", "general", "especifico", "general"],
})

# A) Con dict + map (rápido, sin duplicar filas)
dic = dict(zip(mapeo["origen"], mapeo["tag"]))
largo["tag"] = largo["genero"].map(dic)

# B) Con merge (permite traer varias columnas de la tabla de mapeo)
con_tag = largo.drop(columns="tag").merge(mapeo, left_on="genero", right_on="origen", how="left")
con_tag

# Revisar qué géneros NO quedaron mapeados (clave para completar la tabla):
largo.loc[largo["tag"].isna(), "genero"].value_counts()

In [ ]:
# Una tabla de mapeo se puede construir/editar como CSV y leerla con read_csv.
# Agrupar orígenes por tag (para armar el campo "orígenes" de la colección tags):
mapeo.groupby("tag")["origen"].agg(list)

## 8. Agrupar y agregar (construir los stands)
Una fila por **productor**, juntando los tags de todo su catálogo.

In [ ]:
base = con_tag.copy()
base["Publisher"] = base["Name"].map({"Alpha": "acme", "Beta": "acme", "Gamma": "zeta co", "Epsilon": "omega"})
base = base.dropna(subset=["Publisher"])

# Agregación nombrada: una fila por productor
stands = (base.groupby("Publisher")
              .agg(n_titulos=("Name", "nunique"),
                   tags=("tag", lambda s: sorted(set(s.dropna()))))   # set => sin repetidos
              .reset_index())
stands

In [ ]:
# Otras formas de agrupar
base.groupby("Publisher").size()                     # filas por grupo
base.groupby("Publisher")["tag"].nunique()           # distintos por grupo
base.groupby(["Publisher", "tag"]).size().unstack(fill_value=0)   # tabla cruzada
pd.crosstab(base["Publisher"], base["tag"])          # igual, más directo
base.groupby("Publisher")["tag"].agg(["count", "nunique"])
base.groupby("Publisher")["tag"].transform("count")  # valor del grupo en cada fila
base["tag"].map(base["tag"].value_counts())          # frecuencia del tag pegada a cada fila
base.pivot_table(index="Publisher", columns="tag", values="Name", aggfunc="count", fill_value=0)

## 9. Unir tablas

In [ ]:
a = pd.DataFrame({"k": [1, 2, 3], "x": ["a", "b", "c"]})
b = pd.DataFrame({"k": [2, 3, 4], "y": ["B", "C", "D"]})

a.merge(b, on="k", how="inner")    # solo coincidentes
a.merge(b, on="k", how="left")     # todo de a
a.merge(b, on="k", how="outer", indicator=True)   # todo; _merge dice de dónde vino
a.merge(b, left_on="k", right_on="k", suffixes=("_a", "_b"))
a.set_index("k").join(b.set_index("k"), how="left")

# Apilar tablas con las mismas columnas (ej: stands de los 3 dominios):
s1 = pd.DataFrame({"nombre": ["A"], "dominio": ["juego"]})
s2 = pd.DataFrame({"nombre": ["B"], "dominio": ["anime"]})
pd.concat([s1, s2], ignore_index=True)
pd.concat([s1, s2], keys=["f1", "f2"])           # conserva de qué tabla vino

# .assign agrega columnas sin modificar el original; .pipe encadena funciones
s1.assign(fuente="vgsales").pipe(lambda d: d.rename(columns={"nombre": "stand"}))

## 10. Frecuencias, IDF y ordenar

In [ ]:
frec = pd.DataFrame({"tag": ["accion", "rol", "terror"], "frecuencia": [40, 10, 2]})
N = 50  # total de stands (ejemplo)

frec["idf"] = np.log(N / frec["frecuencia"])      # tags raros pesan más
frec["idf_suave"] = np.log((1 + N) / (1 + frec["frecuencia"])) + 1
frec.sort_values("idf", ascending=False)

# Ordenar y rankear
frec.sort_values(["frecuencia", "tag"], ascending=[False, True])
frec.nlargest(2, "frecuencia"); frec.nsmallest(2, "frecuencia")
frec["ranking"] = frec["frecuencia"].rank(ascending=False, method="dense")
frec.round(3)

In [ ]:
# Jaccard con conjuntos de Python (pandas solo aplica la función a cada fila)
def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if a | b else 0.0

gustos = ["accion", "rol"]
catalogo = pd.DataFrame({"stand": ["S1", "S2", "S3"],
                         "tags": [["accion"], ["rol", "accion", "terror"], ["terror"]]})
catalogo["sim"] = catalogo["tags"].map(lambda t: jaccard(gustos, t))
catalogo.sort_values("sim", ascending=False)

## 11. Datos sintéticos (asistentes e interacciones)

In [ ]:
rng = np.random.default_rng(42)          # semilla => resultados reproducibles

tags_posibles = ["accion", "aventura", "rol", "terror", "ciencia-ficcion"]
stands_ids = [f"stand_{i}" for i in range(5)]

# Asistentes: hasta 8 gustos declarados
asistentes = pd.DataFrame({
    "_id": [f"asis_{i}" for i in range(4)],
    "intereses_declarados": [list(rng.choice(tags_posibles, size=rng.integers(1, 4), replace=False))
                             for _ in range(4)],
})

# Interacciones: muestreo con reemplazo
interacciones = pd.DataFrame({
    "asistente_id": rng.choice(asistentes["_id"], size=10),
    "objetivo_id":  rng.choice(stands_ids, size=10),
    "tipo":         rng.choice(["visita", "asistencia", "favorito"], size=10, p=[0.6, 0.25, 0.15]),
    "fecha":        pd.Timestamp("2026-11-01 10:00") + pd.to_timedelta(rng.integers(0, 600, size=10), unit="m"),
})
interacciones.sort_values("fecha").head()

In [ ]:
# Más herramientas útiles
catalogo.sample(2, random_state=1)           # filas al azar
catalogo.sample(frac=0.5, random_state=1)    # fracción
pd.date_range("2026-11-01 10:00", periods=4, freq="2h")   # horarios de presentaciones
pd.Series(pd.date_range("2026-11-01 10:00", periods=3, freq="h")).dt.hour   # .dt: hora, día, etc.

# Contadores "tag -> cantidad" por asistente (intereses_deducidos):
fav = pd.DataFrame({"asistente_id": ["a1", "a1", "a2"],
                    "tag": ["accion", "accion", "rol"]})
fav.groupby(["asistente_id", "tag"]).size()                       # serie con 2 niveles
(fav.groupby("asistente_id")["tag"].value_counts()
    .groupby(level=0).apply(lambda s: s.droplevel(0).to_dict()))  # {asistente: {tag: n}}

## 12. Salida: guardar y preparar para MongoDB

In [ ]:
OUT = Path("datos") / "interim"   # carpeta de trabajo (no se crea aquí)
# OUT.mkdir(parents=True, exist_ok=True)

# toy.to_csv(OUT / "toy.csv", index=False)
# toy.to_parquet(OUT / "toy.parquet")        # conserva tipos (requiere pyarrow)
# toy.to_json(OUT / "toy.json", orient="records", lines=True, force_ascii=False)

# Preparar documentos para Mongo:
docs_df = stands.copy()

# 1) NaN no existe en JSON/BSON -> convertir a None
docs_df = docs_df.astype(object).where(docs_df.notna(), None)

# 2) DataFrame -> lista de diccionarios
docs = docs_df.to_dict(orient="records")
docs[:2]

In [ ]:
# 3) Tipos numpy -> tipos Python (pymongo no acepta numpy.int64 directamente)
docs = [{k: (v.item() if hasattr(v, "item") else v) for k, v in d.items()} for d in docs]

# 4) Carga con pymongo (ejemplo, no se ejecuta):
# from pymongo import MongoClient
# cli = MongoClient("mongodb://localhost:27017")
# db = cli["gamecon"]
# db.stands.insert_many(docs)               # para listas grandes: insertar por lotes
# db.stands.create_index("tags")            # índices que decidan para sus 4 consultas
# pd.DataFrame(list(db.stands.find({"tags": {"$in": ["accion"]}})))   # leer de vuelta a pandas

## 13. Mapa: qué función va en qué paso
| Paso del proyecto | Funciones de pandas |
|---|---|
| Cargar 3 fuentes | `read_csv`, `read_json(lines=True)`, `usecols`, `nrows`, `chunksize` |
| Perfilar | `shape`, `dtypes`, `info`, `describe`, `isna().sum()`, `duplicated`, `nunique`, `value_counts` |
| Quedarse con lo útil | `[[cols]]`, `drop`, `rename`, `dropna(subset=)`, `query`, `isin` |
| Normalizar géneros (D2) | `.str.strip/lower/replace`, `unicodedata` + `map`, `split`, `explode` |
| Tabla de mapeo | `dict` + `map`, `merge(how='left')`, `isna` para ver faltantes |
| Stands por productor (D1) | `groupby().agg(..., lambda s: sorted(set(s)))`, `assign(dominio=)`, `concat` |
| Colección `tags` | `value_counts` (frecuencia), `groupby('tag')['origen'].agg(list)`, `np.log` (IDF) |
| Presentaciones | `date_range`, `to_datetime`, `sample`, `assign` |
| Asistentes / interacciones | `numpy.random.default_rng`, `choice`, `DataFrame`, `sample`, `to_timedelta` |
| Perfil deducido (D3) | `groupby([asistente, tag]).size()`, `value_counts`, `to_dict` |
| Exportar / cargar | `to_csv`, `to_parquet`, `to_dict('records')`, `where(notna, None)`, `pymongo.insert_many` |

### Errores típicos
- **`SettingWithCopyWarning`**: usa `.copy()` después de filtrar, antes de modificar columnas.
- **Comparar con `NaN`**: `x == np.nan` siempre es falso; usa `isna()` / `notna()`.
- **`str.contains` con nulos**: agrega `na=False` o el filtro falla.
- **`merge` que multiplica filas**: revisa `len()` antes y después; claves duplicadas en la tabla derecha son la causa.
- **`explode` + `groupby`**: recuerden volver a `agg(list)`/`set` para reconstruir las listas por stand.
- **Normalizar antes de agrupar**: si no, `"Acme"` y `"acme"` serían dos stands distintos.